# Bivariate Analysis: RAS vs. NFL Success

In [20]:
import pandas as pd
import numpy as np

In [21]:
# Load pre-merged dataset (using right join since it is based on the RAS dataset)
df_merged = pd.read_csv("../data/cleaned/merged_ras_base_right_join.csv")

# Convert metrics to numeric and fill NaNs where appropriate
df_merged['RAS'] = pd.to_numeric(df_merged['RAS'], errors='coerce')
df_merged['g'] = pd.to_numeric(df_merged['g'], errors='coerce').fillna(0)
df_merged['years_as_primary_starter'] = pd.to_numeric(df_merged['years_as_primary_starter'], errors='coerce').fillna(0)
df_merged['draft_round'] = pd.to_numeric(df_merged['draft_round'], errors='coerce').fillna(0)
df_merged['pro_bowls'] = pd.to_numeric(df_merged['pro_bowls'], errors='coerce').fillna(0)
df_merged['all_pros_first_team'] = pd.to_numeric(df_merged['all_pros_first_team'], errors='coerce').fillna(0)

# Create binary accolade flag
df_merged['has_accolade'] = ((df_merged['pro_bowls'] > 0) | (df_merged['all_pros_first_team'] > 0)).astype(int)

# Create Weighted Success Index (WSI)
df_merged['weighted_success_index'] = (
    (df_merged['years_as_primary_starter'] * 1.0) +
    (df_merged['pro_bowls'] * 3.0) +
    (df_merged['all_pros_first_team'] * 6.0)
)

# Keep only players that successfully matched between both datasets and have valid RAS scores
df_analysis = df_merged.dropna(subset=['name', 'RAS']).copy()

# Remove Special Teams (ST) players
df_analysis = df_analysis[df_analysis['simple_pos'] != 'ST'].copy()

print(f"Total players with both RAS and NFL draft career data (excluding Special Teams): {df_analysis.shape[0]}")

Total players with both RAS and NFL draft career data (excluding Special Teams): 6450


Now that the pre-merged dataset is loaded, we perform a bivariate analysis to measure the relationship between a player's Relative Athletic Score (RAS) and their NFL success.

### Hypotheses & Methodology:
1. **Overall Relationship**: Does raw athleticism correlate with NFL success? We will calculate **Pearson** and **Spearman Rank** correlations between RAS and multiple success metrics:
   - `g` (Games Played) - proxy for career longevity/durability.
   - `years_as_primary_starter` - proxy for starter-level value (our standardized proxy for games started).
   - Individual accolades: `pro_bowls` and `all_pros_first_team`.
   - **Weighted Success Index (WSI)**: Our custom weighted score combining longevity and accolades:
     $$\\text{WSI} = \\text{Starter Years} + (3 \\times \\text{Pro Bowls}) + (6 \\times \\text{First-Team All-Pros})$$
2. **Necessary Condition (Skill Positions)**: We group by position (excluding Special Teams) to determine if a high RAS score is a necessary condition for success in "skill" positions (Wide Receivers, Defensive Backs, Running Backs, Tight Ends) compared to size/technique positions (Offensive and Defensive Linemen).

### 1. Overall Bivariate Correlation

In [22]:
# Compute Pearson and Spearman Rank correlation (Spearman via pandas rank() to avoid scipy dependency)
corr_g_pearson = df_analysis['RAS'].corr(df_analysis['g'], method='pearson')
corr_g_spearman = df_analysis['RAS'].rank().corr(df_analysis['g'].rank(), method='pearson')
corr_start_pearson = df_analysis['RAS'].corr(df_analysis['years_as_primary_starter'], method='pearson')
corr_start_spearman = df_analysis['RAS'].rank().corr(df_analysis['years_as_primary_starter'].rank(), method='pearson')
corr_wsi_pearson = df_analysis['RAS'].corr(df_analysis['weighted_success_index'], method='pearson')
corr_wsi_spearman = df_analysis['RAS'].rank().corr(df_analysis['weighted_success_index'].rank(), method='pearson')

print(f"Correlation between RAS and Games Played (Pearson): {corr_g_pearson:.3f}")
print(f"Correlation between RAS and Games Played (Spearman): {corr_g_spearman:.3f}")
print(f"Correlation between RAS and Years as Primary Starter (Pearson): {corr_start_pearson:.3f}")
print(f"Correlation between RAS and Years as Primary Starter (Spearman): {corr_start_spearman:.3f}")
print(f"Correlation between RAS and Weighted Success Index (Pearson): {corr_wsi_pearson:.3f}")
print(f"Correlation between RAS and Weighted Success Index (Spearman): {corr_wsi_spearman:.3f}")

Correlation between RAS and Games Played (Pearson): 0.119
Correlation between RAS and Games Played (Spearman): 0.138
Correlation between RAS and Years as Primary Starter (Pearson): 0.120
Correlation between RAS and Years as Primary Starter (Spearman): 0.143
Correlation between RAS and Weighted Success Index (Pearson): 0.111
Correlation between RAS and Weighted Success Index (Spearman): 0.148


### 2. Grouping & Aggregation by Player Position

We group the dataset by position (`simple_pos`) and calculate:
- Pearson and Spearman correlations between RAS and our Weighted Success Index (WSI) within each position.
- Average RAS for "successful" players (defined as WSI $\ \ge 5.0$, representing at least 5 years as a starter or a combination of starting and accolades) vs other players.
- The percentage of successful players who possess a high RAS score ($\ \ge 7.0$).

In [23]:
pos_group = df_analysis.groupby('simple_pos')
pos_analysis = []

for name, group in pos_group:
    if len(group) < 30:  # Skip small sample sizes
        continue
    p_corr_wsi = group['RAS'].corr(group['weighted_success_index'], method='pearson')
    s_corr_wsi = group['RAS'].rank().corr(group['weighted_success_index'].rank(), method='pearson')
    p_corr_g = group['RAS'].corr(group['g'], method='pearson')
    
    # Define success as WSI >= 5.0 (equivalent to 5 seasons starting or a combination of starting & accolades)
    group_success = group[group['weighted_success_index'] >= 5.0]
    group_other = group[group['weighted_success_index'] < 5.0]
    
    avg_ras_success = group_success['RAS'].mean() if len(group_success) > 0 else np.nan
    avg_ras_other = group_other['RAS'].mean() if len(group_other) > 0 else np.nan
    
    # Calculate % of players with high RAS (>= 7.0)
    pct_success_high_ras = (group_success['RAS'] >= 7.0).mean() * 100 if len(group_success) > 0 else np.nan
    pct_other_high_ras = (group_other['RAS'] >= 7.0).mean() * 100 if len(group_other) > 0 else np.nan
    
    pos_analysis.append({
        'Position': name,
        'Count': len(group),
        'Pearson (RAS vs WSI)': p_corr_wsi,
        'Spearman (RAS vs WSI)': s_corr_wsi,
        'Pearson (RAS vs Games)': p_corr_g,
        'Success Count (WSI>=5)': len(group_success),
        'Avg RAS (Successful)': avg_ras_success,
        'Avg RAS (Other)': avg_ras_other,
        'Success with RAS>=7.0 (%)': pct_success_high_ras,
        'Other with RAS>=7.0 (%)': pct_other_high_ras
    })

df_pos_analysis = pd.DataFrame(pos_analysis)
print(df_pos_analysis.to_string(index=False))

Position  Count  Pearson (RAS vs WSI)  Spearman (RAS vs WSI)  Pearson (RAS vs Games)  Success Count (WSI>=5)  Avg RAS (Successful)  Avg RAS (Other)  Success with RAS>=7.0 (%)  Other with RAS>=7.0 (%)
      DB   1296              0.092019               0.153313                0.127248                     269              7.551599         7.054411                  66.542751                58.714703
      DL   1070              0.136246               0.114387                0.107259                     245              7.142653         6.525964                  60.000000                50.181818
      LB    780              0.135285               0.194000                0.191519                     188              7.674628         6.864544                  69.680851                57.263514
      OL   1162              0.138451               0.173427                0.126328                     333              7.370511         6.560374                  63.663664                51.507841


### 3. Draft Round Analysis

Let's see if RAS is more critical for early round picks (rounds 1-3) compared to late round picks (rounds 4-7).

In [24]:
df_analysis['draft_round_group'] = pd.cut(df_analysis['draft_round'], bins=[0, 3, 7, 12], labels=['Early (1-3)', 'Late (4-7)', 'Other/Undrafted'])
round_group = df_analysis.groupby('draft_round_group', observed=False)
round_analysis = []

for name, group in round_group:
    p_corr_wsi = group['RAS'].corr(group['weighted_success_index'], method='pearson')
    s_corr_wsi = group['RAS'].rank().corr(group['weighted_success_index'].rank(), method='pearson')
    round_analysis.append({
        'Round Group': name,
        'Count': len(group),
        'Pearson (RAS vs WSI)': p_corr_wsi,
        'Spearman (RAS vs WSI)': s_corr_wsi
    })

print(pd.DataFrame(round_analysis).to_string(index=False))

    Round Group  Count  Pearson (RAS vs WSI)  Spearman (RAS vs WSI)
    Early (1-3)   2687              0.083240               0.087806
     Late (4-7)   3763             -0.001383               0.015024
Other/Undrafted      0                   NaN                    NaN


# Interactive Heatmap: Success Metrics by Position and RAS Bin

To visually explore the relationship between athleticism (RAS bins) and career accolades/success metrics across different positions, we construct an interactive heatmap.

The heatmap displays:
- **Y-axis**: Player Position (`simple_pos`)
- **X-axis**: RAS Bins (`0-2`, `2-4`, `4-6`, `6-8`, `8-10`)
- **Cell Value (Color Scale)**: The average value of the selected success metric for players in that position and RAS bin pairing.

Use the dropdown menu to select different success metrics, including Games Played (G), Years as Primary Starter, Pro Bowls, First-Team All-Pros, Accolade Rate (%), and our custom Weighted Success Index (WSI).

In [25]:
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact

In [26]:
def plot_heatmap(metric):
    # Map dropdown options to dataframe columns
    metric_map = {
        'Games Played (G)': 'g',
        'Years as Primary Starter': 'years_as_primary_starter',
        'Pro Bowls': 'pro_bowls',
        'All-Pros (First Team)': 'all_pros_first_team',
        'Accolade Rate (%)': 'has_accolade',
        'Weighted Success Index (WSI)': 'weighted_success_index'
    }
    col = metric_map[metric]
    
    df_plot = df_analysis.copy()
    
    # Fill NaNs with 0 for accolades/metrics
    df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce').fillna(0)
    
    # Bin RAS scores
    df_plot['ras_bin'] = pd.cut(
        df_plot['RAS'], 
        bins=[0, 2, 4, 6, 8, 10], 
        labels=['0-2 (Very Low)', '2-4 (Low)', '4-6 (Medium)', '6-8 (High)', '8-10 (Elite)'], 
        include_lowest=True
    )
    
    # Set labels, formats, and colors based on metric
    if metric == 'Accolade Rate (%)':
        df_plot[col] = df_plot[col] * 100
        fmt_str = ".1f"
        cbar_lbl = "Accolade Rate (%)"
        title_lbl = "Accolade Rate (Pro Bowl/All-Pro %)"
        colormap = "viridis"
    elif metric == 'Weighted Success Index (WSI)':
        fmt_str = ".2f"
        cbar_lbl = "Weighted Success Index (WSI)"
        title_lbl = "Average Weighted Success Index (WSI)"
        colormap = "magma"
    else:
        fmt_str = ".2f"
        cbar_lbl = f"Average {metric}"
        title_lbl = f"Average {metric}"
        colormap = "plasma"
    
    # Group and pivot
    pivot_df = df_plot.pivot_table(index='simple_pos', columns='ras_bin', values=col, aggfunc='mean', observed=False)
    
    # Ensure correct column ordering
    cols_order = ['0-2 (Very Low)', '2-4 (Low)', '4-6 (Medium)', '6-8 (High)', '8-10 (Elite)']
    pivot_df = pivot_df[cols_order]
    
    # Render Heatmap with premium dark/vibrant styling
    sns.set_theme(style="whitegrid", palette="muted")
    fig = plt.figure(figsize=(12, 7.5))
    
    ax = sns.heatmap(pivot_df, annot=True, fmt=fmt_str, cmap=colormap, cbar_kws={'label': cbar_lbl}, 
                     linewidths=0.5, linecolor='white', annot_kws={'size': 11, 'weight': 'bold'})
    
    plt.title(f"NFL Success Heatmap: {title_lbl} by Position and RAS Bin", fontsize=15, pad=15, weight='bold')
    plt.ylabel("Position (simple_pos)", fontsize=12, labelpad=10)
    plt.xlabel("RAS Score Bin", fontsize=12, labelpad=10)
    plt.xticks(fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()

# Construct and render interactive widget
style = {'description_width': 'initial'}
metric_dropdown = widgets.Dropdown(
    options=[
        'Games Played (G)',
        'Years as Primary Starter',
        'Pro Bowls',
        'All-Pros (First Team)',
        'Accolade Rate (%)',
        'Weighted Success Index (WSI)'
    ],
    value='Weighted Success Index (WSI)',
    description='Select Success Metric:',
    style=style
)

interact(plot_heatmap, metric=metric_dropdown);

interactive(children=(Dropdown(description='Select Success Metric:', index=5, options=('Games Played (G)', 'Ye…